# Drift Evaluation Completo (Sintético + Estado + Episodios)

Notebook listo para **descargar y ejecutar** que implementa:

- Carga de datos sintéticos (`synthetic_plant.csv`) y etiquetado manual (`synthetic_plant_events.csv`).

- Detección de drift con estado por variable (`NORMAL` / `DRIFT`) a nivel de ventanas.
- Referencia que se recalcula en cada ventana según la estrategia (`decay`, `golden`, `seasonal`).
- Construcción de episodios automáticos a partir de secuencias de ventanas con drift.

- Exportación de CSVs con episodios automáticos, evaluación de episodios y métricas agregadas.

## 1. Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib.util
import plotly.express as px

## 2. Cargar módulo `Funciones_Drift.py` y alias útiles

In [2]:
spec = importlib.util.spec_from_file_location("funciones_drift", "../Analisis/Funciones_Drift.py")
funciones_drift = importlib.util.module_from_spec(spec)
spec.loader.exec_module(funciones_drift)

# Aliases a funciones que usaremos
strip_outliers         = funciones_drift.strip_outliers
ref_decay_prefix_mass  = funciones_drift.ref_decay_prefix_mass
ref_golden             = funciones_drift.ref_golden
ref_seasonal           = funciones_drift.ref_seasonal
score_numeric_series   = funciones_drift._score_numeric_series

## 3. Carga de serie sintética y construcción de intervalos manuales

In [3]:
# Paths de entrada
SERIES_PATH = Path("synthetic_data/synthetic_plant.csv")
LABELS_PATH = Path("synthetic_data/synthetic_plant_events.csv")

assert SERIES_PATH.exists(), f"No se encontró {SERIES_PATH}"
assert LABELS_PATH.exists(),  f"No se encontró {LABELS_PATH}"

# --- Serie sintética ---
df_raw = pd.read_csv(SERIES_PATH)
# Detectar columna de tiempo si no se llama exactamente 'date_time'
if "date_time" not in df_raw.columns:
    for c in ["datetime","timestamp","time","fecha","tiempo"]:
        if c in df_raw.columns:
            df_raw = df_raw.rename(columns={c:"date_time"})
            break

df_raw["date_time"] = pd.to_datetime(df_raw["date_time"], errors="coerce")
df_raw = (df_raw
          .dropna(subset=["date_time"])
          .sort_values("date_time")
          .set_index("date_time"))

# Quitamos outliers usando tu helper
df_raw = strip_outliers(df_raw)

# Nos quedamos con columnas numéricas (las típicas var_1, var_2, etc.)
df = df_raw.select_dtypes(include="number").copy()
assert not df.empty, "No hay columnas numéricas en la serie sintética."

t_min, t_max = df.index.min(), df.index.max()

# --- Etiquetado manual ---
events = pd.read_csv(LABELS_PATH)
events["date_time"] = pd.to_datetime(events["date_time"], errors="coerce")
events = (events
          .dropna(subset=["date_time","variable","event"])
          .assign(event=lambda s: s["event"].str.lower().str.strip())
          .query("event in ['start','end']")
          .sort_values(["variable","date_time"])
          .reset_index(drop=True))

def events_to_intervals(ev: pd.DataFrame) -> pd.DataFrame:
    """
    Convierte pares start/end por variable en episodios continuos (manuales).
    Si existe columna 'drift_type' en el CSV de eventos, la conserva por episodio.
    Se asume que start y end de un mismo episodio tienen el mismo drift_type.
    """
    has_type = "drift_type" in ev.columns
    rows = []
    for var, g in ev.groupby("variable", sort=True):
        open_t = None
        open_type = None
        for _, r in g.iterrows():
            evt = str(r["event"]).lower()
            dt_val = r["drift_type"] if has_type else "unknown"

            if evt == "start":
                open_t = r["date_time"]
                open_type = dt_val
            elif evt == "end" and open_t is not None and r["date_time"] > open_t:
                rows.append({
                    "variable": var,
                    "manual_start": open_t,
                    "manual_end": r["date_time"],
                    "drift_type": open_type,
                })
                open_t = None
                open_type = None
    return pd.DataFrame(rows)

intervals_manual = events_to_intervals(events)

## 4. Motor stateful de detección de drift (PSI + referencia congelada + episodios)

In [4]:
def run_drift_for_strategy_multi_metric(
    df: pd.DataFrame,
    window: str,
    strategy: str,
    metrics: tuple = ("psi", "ks", "wasserstein"),
    thresholds: dict | None = None,
    min_points: int = 5,
):
    """
    Ejecuta detección de drift para una estrategia dada y *varias* métricas numéricas.

    - NO hay referencia congelada.
    - En cada ventana se construye la referencia según `strategy` usando TODO el historial hasta t0.
    - Para cada métrica en `metrics`:
        stat_value = score_numeric_series(ref, cur, metric)
        drift_flag = stat_value >= threshold_métrica
    - Un episodio es una secuencia contigua de ventanas con drift_flag=True.
    - Cuando la métrica baja del umbral, se cierra el episodio y listo; la referencia
      en pasos futuros ya incorpora el nuevo nivel estable.

    `thresholds`: dict opcional {metric_name: valor}. Si no se pasa, usa los defaults
    coherentes con tu Funciones_Drift:

        psi         ~ 0.2
        ks          ~ 0.15
        wasserstein -> 0.5 * std(ref) si no se fija
    """

    # Defaults coherentes con _dispatch_build_metrics
    default_thr = {"psi": 0.2, "ks": 0.15, "wasserstein": np.nan}
    thresholds = thresholds or {}

    w = pd.to_timedelta(window)
    t_min, t_max = df.index.min(), df.index.max()
    t_ends = pd.date_range(t_min + w, t_max, freq=window)

    variables = list(df.columns)

    # Estado por (métrica, variable) sólo para numerar episodios contiguos
    state = {
        metric: {var: "NORMAL" for var in variables}
        for metric in metrics
    }
    current_episode = {
        metric: {var: 0 for var in variables}
        for metric in metrics
    }

    rows = []

    for t_end in t_ends:
        t0 = t_end - w

        df_hist = df.loc[: t0 - pd.Timedelta(microseconds=1)]
        df_cur  = df.loc[t0:t_end]

        if df_hist.empty or df_cur.empty:
            continue

        # Referencia según la estrategia (SIEMPRE recalculada)
        if strategy == "decay":
            ref_global = ref_decay_prefix_mass(df_hist, now=t_end)
        elif strategy == "golden":
            ref_global = ref_golden(df_hist)
        elif strategy == "seasonal":
            ref_global = ref_seasonal(df_hist, current_end=t_end)
        else:
            raise ValueError(f"Estrategia desconocida: {strategy}")

        if ref_global is None or ref_global.empty:
            ref_global = df_hist

        for var in variables:
            cur_series = df_cur[var].dropna()

            if cur_series.size < min_points:
                # Sin datos suficientes: logueamos filas sin drift (no cambiamos estado)
                for metric_name in metrics:
                    rows.append({
                        "variable": var,
                        "strategy": strategy,
                        "window": window,
                        "metric": metric_name,
                        "t0": t0,
                        "t1": t_end,
                        "drift_flag": False,
                        "episode_id": np.nan,
                        "stat_value": None,
                        "threshold": None,
                        "state": state[metric_name][var],
                    })
                continue

            # Referencia para esta variable
            if var in ref_global.columns:
                ref_series = ref_global[var].dropna()
            else:
                ref_series = df_hist[var].dropna()

            for metric_name in metrics:
                base_thr = default_thr.get(metric_name, 0.2)
                thr = thresholds.get(metric_name, base_thr)

                if ref_series.empty:
                    stat_val = None
                    eff_thr = thr
                    drift_flag = False
                else:
                    # Usa exactamente la misma función que el pipeline
                    stat_val = score_numeric_series(ref_series, cur_series, metric_name)

                    # Para Wasserstein, si el threshold viene como NaN, lo adaptamos a la escala ref
                    eff_thr = thr
                    if metric_name == "wasserstein" and (eff_thr is None or (isinstance(eff_thr, float) and np.isnan(eff_thr))):
                        std_ref = pd.to_numeric(ref_series, errors="coerce").dropna().std()
                        eff_thr = float(std_ref) * 0.5 if pd.notna(std_ref) else 0.5

                    if stat_val is None or np.isnan(stat_val):
                        drift_flag = False
                    else:
                        drift_flag = bool(stat_val >= eff_thr)

                # Actualizar estado/episodios (sin referencia congelada)
                if drift_flag:
                    if state[metric_name][var] == "NORMAL":
                        current_episode[metric_name][var] += 1
                        state[metric_name][var] = "DRIFT"
                else:
                    if state[metric_name][var] == "DRIFT":
                        state[metric_name][var] = "NORMAL"

                rows.append({
                    "variable": var,
                    "strategy": strategy,
                    "window": window,
                    "metric": metric_name,
                    "t0": t0,
                    "t1": t_end,
                    "drift_flag": drift_flag,
                    "episode_id": (
                        current_episode[metric_name][var]
                        if drift_flag else np.nan
                    ),
                    "stat_value": stat_val,
                    "threshold": eff_thr,
                    "state": state[metric_name][var],
                })

    return pd.DataFrame(rows)


def run_drift_all_multi_metric(
    df: pd.DataFrame,
    windows=("6H","12H","24H","48H"),
    strategies=("decay","golden","seasonal"),
    metrics: tuple = ("psi", "ks", "wasserstein"),
    thresholds: dict | None = None,
    min_points: int = 5,
):
    """Runner multi-ventana, multi-estrategia y multi-métrica."""
    all_frames = []
    for win in windows:
        for strat in strategies:
            dfw = run_drift_for_strategy_multi_metric(
                df=df,
                window=win,
                strategy=strat,
                metrics=metrics,
                thresholds=thresholds,
                min_points=min_points,
            )
            all_frames.append(dfw)

    if not all_frames:
        return pd.DataFrame()
    return pd.concat(all_frames, ignore_index=True)

In [5]:
EVAL_WINDOWS = ["2H","4H", "6H", "12H", "24H", "36H", "48H"]
STRATEGIES   = ["decay","golden","seasonal"]
METRICS      = ("psi", "ks", "wasserstein")

# Si quieres tuneo fino de thresholds por métrica, lo puedes hacer aquí
METRIC_THRESHOLDS = {
    # si no defines alguno, se usan los defaults del motor (psi~0.2, ks~0.15)
    # "psi": 0.2,
    # "ks": 0.15,
    # "wasserstein": np.nan,   # → se adaptará a 0.5 * std(ref)
}

df_windows = run_drift_all_multi_metric(
    df=df,
    windows=EVAL_WINDOWS,
    strategies=STRATEGIES,
    metrics=METRICS,
    thresholds=METRIC_THRESHOLDS,
    min_points=5,
)

## 5. Compactar ventanas en episodios automáticos

In [6]:
def windows_to_episodes_multi_metric(df_windows: pd.DataFrame) -> pd.DataFrame:
    """
    Compacta secuencias de ventanas con drift_flag=1
    en episodios automáticos por (window, strategy, metric, variable, episode_id).
    """
    dfw = df_windows.copy()
    dfw = dfw[dfw["drift_flag"] == True].dropna(subset=["episode_id"])
    if dfw.empty:
        return pd.DataFrame(columns=[
            "window","strategy","metric","variable","episode_id",
            "seg_start","seg_end","seg_length","stat_max"
        ])

    rows = []
    for keys, sub in dfw.groupby(
        ["window","strategy","metric","variable","episode_id"],
        dropna=False
    ):
        win, strat, metric, var, eid = keys
        sub = sub.sort_values("t0")
        seg_start = sub["t0"].min()
        seg_end   = sub["t1"].max()
        stat_max  = sub["stat_value"].max()
        rows.append({
            "window": win,
            "strategy": strat,
            "metric": metric,
            "variable": var,
            "episode_id": int(eid),
            "seg_start": seg_start,
            "seg_end": seg_end,
            "seg_length": seg_end - seg_start,
            "stat_max": stat_max,
        })
    return pd.DataFrame(rows)

df_episodes_auto = windows_to_episodes_multi_metric(df_windows)
print("Episodios automáticos:", len(df_episodes_auto))

Episodios automáticos: 4907


## 6. Evaluación episodio-a-episodio vs etiquetado manual

In [14]:
def evaluate_episodes_vs_manual_multi_metric(
    df_episodes_auto: pd.DataFrame,
    intervals_manual: pd.DataFrame,
):
    """
    Compara episodios automáticos vs episodios manuales por (window, strategy, metric):

      Vista por EPISODIOS (como ya tenías):
        - TP_episodes: episodios manuales con al menos un solape con algún episodio auto.
        - FN_episodes: episodios manuales sin solape.
        - FP_episodes: episodios auto sin solape con ningún episodio manual.
        - Precision, Recall, F1: métricas por conteo de episodios.

      Vista por TIEMPO (nuevo):
        - manual_total_hours: horas totales en drift manual.
        - auto_total_hours: horas totales en drift automático.
        - overlap_hours: horas en que ambos están en drift.
        - Precision_time, Recall_time, F1_time: métricas basadas en longitudes de intervalos.
    """

    if df_episodes_auto.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    man = intervals_manual.copy()

    # Duración total de la serie (para tasas por día)
    total_days = max((t_max - t_min).total_seconds() / (3600 * 24), 1e-9)

    results = []
    marks_manual_all = []
    marks_auto_all   = []

    for (win, strat, metric), auto_sub in df_episodes_auto.groupby(
        ["window", "strategy", "metric"], dropna=False
    ):
        vars_in_auto = sorted(auto_sub["variable"].unique())
        man_sub = man[man["variable"].isin(vars_in_auto)].copy()
        if man_sub.empty and auto_sub.empty:
            continue

        # -----------------------------
        # 1) Vista manual: TP/FN + coverage + delay (tu lógica)
        # -----------------------------
        manual_matches = []
        coverage_vals  = []
        delay_vals     = []

        for _, mrow in man_sub.iterrows():
            v  = mrow["variable"]
            ms = mrow["manual_start"]
            me = mrow["manual_end"]

            rel_auto = auto_sub[auto_sub["variable"] == v]

            total_overlap = pd.Timedelta(0)
            first_det = None

            for _, arow in rel_auto.iterrows():
                as_ = arow["seg_start"]
                ae  = arow["seg_end"]

                # solape con el episodio manual
                start = max(ms, as_)
                end   = min(me, ae)
                if end > start:
                    total_overlap += (end - start)

                    # tiempo de detección (clamp en el inicio manual)
                    det_candidate = as_
                    if det_candidate < ms:
                        det_candidate = ms
                    if first_det is None or det_candidate < first_det:
                        first_det = det_candidate

            matched = total_overlap > pd.Timedelta(0)
            manual_matches.append(bool(matched))

            # coverage
            dur = me - ms
            if dur.total_seconds() > 0:
                cov_val = total_overlap.total_seconds() / dur.total_seconds()
            else:
                cov_val = 0.0
            coverage_vals.append(cov_val)

            # delay en horas (solo si hubo detección)
            if first_det is None:
                delay_vals.append(np.nan)
            else:
                delay = (first_det - ms).total_seconds() / 3600.0
                if delay < 0:
                    delay = 0.0
                delay_vals.append(delay)

        man_sub["matched_auto"] = manual_matches
        man_sub["coverage"]     = coverage_vals
        man_sub["delay_hours"]  = delay_vals

        TP = int(man_sub["matched_auto"].sum()) if not man_sub.empty else 0
        FN = int((~man_sub["matched_auto"]).sum()) if not man_sub.empty else 0

        # -----------------------------
        # 2) Vista auto: FP (tu lógica)
        # -----------------------------
        auto_sub = auto_sub.copy()
        auto_matches = []
        for _, arow in auto_sub.iterrows():
            v  = arow["variable"]
            as_ = arow["seg_start"]
            ae  = arow["seg_end"]

            overlap = (
                (man_sub["variable"] == v) &
                ~(man_sub["manual_end"] < as_) &
                ~(man_sub["manual_start"] > ae)
            ).any()
            auto_matches.append(overlap)

        auto_sub["matched_manual"] = auto_matches
        FP = int((~auto_sub["matched_manual"]).sum()) if not auto_sub.empty else 0

        # -----------------------------
        # 3) Métricas por EPISODIOS (pequeño fix en F1)
        # -----------------------------
        prec = TP / (TP + FP) if (TP + FP) > 0 else np.nan
        rec  = TP / (TP + FN) if (TP + FN) > 0 else np.nan

        if np.isnan(prec) or np.isnan(rec) or (prec + rec) == 0:
            f1 = np.nan
        else:
            f1 = 2 * prec * rec / (prec + rec)

        # coverage y delay agregados
        coverage_mean   = float(np.nanmean(coverage_vals))  if coverage_vals else np.nan
        coverage_median = float(np.nanmedian(coverage_vals)) if coverage_vals else np.nan

        delay_valid = [d for d in delay_vals if not np.isnan(d)]
        delay_mean_hours   = float(np.mean(delay_valid))   if delay_valid else np.nan
        delay_median_hours = float(np.median(delay_valid)) if delay_valid else np.nan

        false_alarms_per_day = FP / total_days

        # -----------------------------
        # 4) Métricas por TIEMPO (nuevo)
        # -----------------------------
        # longitudes totales de episodios
        if not man_sub.empty:
            man_sub["manual_len_sec"] = (
                man_sub["manual_end"] - man_sub["manual_start"]
            ).dt.total_seconds()
            manual_len_total_sec = man_sub["manual_len_sec"].sum()
        else:
            manual_len_total_sec = 0.0

        if not auto_sub.empty:
            auto_sub["auto_len_sec"] = (
                auto_sub["seg_end"] - auto_sub["seg_start"]
            ).dt.total_seconds()
            auto_len_total_sec = auto_sub["auto_len_sec"].sum()
        else:
            auto_len_total_sec = 0.0

        # solape total manual-auto (en segundos)
        overlap_total_sec = 0.0
        if manual_len_total_sec > 0 and auto_len_total_sec > 0:
            for _, mrow in man_sub.iterrows():
                v  = mrow["variable"]
                ms = mrow["manual_start"]
                me = mrow["manual_end"]

                rel_auto = auto_sub[auto_sub["variable"] == v]
                for _, arow in rel_auto.iterrows():
                    as_ = arow["seg_start"]
                    ae  = arow["seg_end"]
                    start = max(ms, as_)
                    end   = min(me, ae)
                    if end > start:
                        overlap_total_sec += (end - start).total_seconds()

        if auto_len_total_sec > 0:
            prec_time = overlap_total_sec / auto_len_total_sec
        else:
            prec_time = np.nan

        if manual_len_total_sec > 0:
            rec_time = overlap_total_sec / manual_len_total_sec
        else:
            rec_time = np.nan

        if np.isnan(prec_time) or np.isnan(rec_time) or (prec_time + rec_time) == 0:
            f1_time = np.nan
        else:
            f1_time = 2 * prec_time * rec_time / (prec_time + rec_time)

        # Guardamos fila de resultados
        results.append({
            "window": win,
            "strategy": strat,
            "metric": metric,
            # episodios
            "TP_episodes": TP,
            "FP_episodes": FP,
            "FN_episodes": FN,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
            # tiempo
            "manual_total_hours": manual_len_total_sec / 3600.0 if manual_len_total_sec > 0 else 0.0,
            "auto_total_hours": auto_len_total_sec / 3600.0 if auto_len_total_sec > 0 else 0.0,
            "overlap_hours": overlap_total_sec / 3600.0 if overlap_total_sec > 0 else 0.0,
            "Precision_time": prec_time,
            "Recall_time": rec_time,
            "F1_time": f1_time,
            # extras que ya tenías
            "coverage_mean": coverage_mean,
            "coverage_median": coverage_median,
            "delay_mean_hours": delay_mean_hours,
            "delay_median_hours": delay_median_hours,
            "false_alarms_per_day": false_alarms_per_day,
        })

        # Guardamos manual/auto marcados (para análisis por tipo de drift, etc.)
        man_sub["window"]   = win
        man_sub["strategy"] = strat
        man_sub["metric"]   = metric

        auto_sub["window"]   = win
        auto_sub["strategy"] = strat
        auto_sub["metric"]   = metric

        marks_manual_all.append(man_sub)
        marks_auto_all.append(auto_sub)

    eval_df = pd.DataFrame(results)
    manual_marked = pd.concat(marks_manual_all, ignore_index=True) if marks_manual_all else pd.DataFrame()
    auto_marked   = pd.concat(marks_auto_all,   ignore_index=True) if marks_auto_all   else pd.DataFrame()

    return eval_df, manual_marked, auto_marked




eval_episodes_df, manual_marked, auto_marked = evaluate_episodes_vs_manual_multi_metric(
    df_episodes_auto=df_episodes_auto,
    intervals_manual=intervals_manual,
)

## 7. Exportar CSVs de resultados

In [15]:
OUTPUT_DIR = Path("synthetic_data/results")
OUTPUT_DIR.mkdir(exist_ok=True)

eval_path   = OUTPUT_DIR / "eval_episodes_by_window_strategy.csv"
manual_path = OUTPUT_DIR / "manual_marked_episodes.csv"
auto_path   = OUTPUT_DIR / "auto_marked_episodes.csv"

eval_episodes_df.to_csv(eval_path, index=False)
if not manual_marked.empty:
    manual_marked.to_csv(manual_path, index=False)
if not auto_marked.empty:
    auto_marked.to_csv(auto_path, index=False)

print("Guardados:")
print(" -", eval_path)
if manual_marked is not None and not manual_marked.empty:
    print(" -", manual_path)
if auto_marked is not None and not auto_marked.empty:
    print(" -", auto_path)

Guardados:
 - synthetic_data\results\eval_episodes_by_window_strategy.csv
 - synthetic_data\results\manual_marked_episodes.csv
 - synthetic_data\results\auto_marked_episodes.csv


In [16]:
METRICS_DIR = OUTPUT_DIR / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
print(f"📂 Directorio de métricas avanzadas: {METRICS_DIR.resolve()}")

has_cov_mean   = "coverage_mean" in eval_episodes_df.columns
has_delay_mean = "delay_mean_hours" in eval_episodes_df.columns
has_delay_med  = "delay_median_hours" in eval_episodes_df.columns
has_false_rate = "false_alarms_per_day" in eval_episodes_df.columns

# ------------------------------
# 7.1 Calidad en detectar drift
# ------------------------------
quality_cols = ["Precision", "Recall", "F1"]

# nuevas métricas basadas en tiempo
time_cols = [c for c in ["Precision_time", "Recall_time", "F1_time"]
             if c in eval_episodes_df.columns]
quality_cols.extend(time_cols)

if has_cov_mean:
    quality_cols.append("coverage_mean")

quality_group = ["metric", "strategy", "window"]

quality_overall = (
    eval_episodes_df
    .groupby(quality_group)[quality_cols]
    .mean()
    .reset_index()
    .sort_values(["metric", "window", "F1"], ascending=[True, True, False])
)
quality_overall.to_csv(METRICS_DIR / "quality_overall.csv", index=False)
print(f"✅ quality_overall.csv guardado ({len(quality_overall)} filas)")

# ------------------------------
# 7.2 Velocidad de detección
# ------------------------------
if has_delay_mean or has_delay_med:
    delay_cols = []
    if has_delay_mean:
        delay_cols.append("delay_mean_hours")
    if has_delay_med:
        delay_cols.append("delay_median_hours")

    speed_overall = (
        eval_episodes_df
        .groupby(quality_group)[delay_cols]
        .mean()
        .reset_index()
        .sort_values(["metric", "window"], ascending=[True, True])
    )
    speed_overall.to_csv(METRICS_DIR / "speed_overall.csv", index=False)
    print(f"✅ speed_overall.csv guardado ({len(speed_overall)} filas)")
else:
    print("ℹ️ No se encontraron columnas de delay en eval_episodes_df → se omiten CSV de velocidad.")

# ------------------------------
# 7.3 Estabilidad (false alarms)
# ------------------------------
if has_false_rate:
    stability_overall = (
        eval_episodes_df
        .groupby(quality_group)[["false_alarms_per_day"]]
        .mean()
        .reset_index()
        .sort_values(["metric", "window", "false_alarms_per_day"],
                     ascending=[True, True, True])
    )
    stability_overall.to_csv(METRICS_DIR / "stability_overall.csv", index=False)
    print(f"✅ stability_overall.csv guardado ({len(stability_overall)} filas)")
else:
    print("ℹ️ No se encontró 'false_alarms_per_day' en eval_episodes_df → se omite stability_overall.csv")

# ------------------------------
# 7.4 Comportamiento por tipo de drift (gradual / abrupto)
#     (usando episodios manuales marcados)
# ------------------------------
has_manual_type = (
    "drift_type" in manual_marked.columns
    and "matched_auto" in manual_marked.columns
    and "coverage" in manual_marked.columns
    and "delay_hours" in manual_marked.columns
)

if has_manual_type and not manual_marked.empty:
    by_type = (
        manual_marked
        .groupby(["metric", "strategy", "window", "drift_type"])
        .agg(
            n_manual=("manual_start", "size"),
            TP=("matched_auto", "sum"),
            Recall=("matched_auto", "mean"),
            coverage_mean=("coverage", "mean"),
            delay_mean_hours=("delay_hours", "mean"),
            delay_median_hours=("delay_hours", "median"),
        )
        .reset_index()
        .sort_values(["metric","drift_type","window"], ascending=[True, True, True])
    )
    by_type.to_csv(METRICS_DIR / "by_drifttype_manual.csv", index=False)
    print(f"✅ by_drifttype_manual.csv guardado ({len(by_type)} filas)")
else:
    print("ℹ️ No se pudo construir análisis por 'drift_type' (faltan columnas o manual_marked está vacío).")

print("🎯 Export de métricas avanzadas terminado.")

📂 Directorio de métricas avanzadas: C:\Users\frncc\OneDrive - Universidad Católica de Chile\Desktop\UC\2025-2\Proyecto de Grado\Proyecto-Grado\Drift Evaluation\synthetic_data\results\metrics
✅ quality_overall.csv guardado (63 filas)
✅ speed_overall.csv guardado (63 filas)
✅ stability_overall.csv guardado (63 filas)
✅ by_drifttype_manual.csv guardado (126 filas)
🎯 Export de métricas avanzadas terminado.


## 8. Visualización 5x2 (export)

In [17]:
def plot_grid_episodes_matplotlib(
    df,
    df_episodes_auto: pd.DataFrame,
    intervals_manual: pd.DataFrame,
    strategy: str,
    window: str,
    metric: str = "psi",
    show_manual: bool = True,
):
    """
    Versión Matplotlib de la grilla 5x2:
      - Línea de la serie.
      - Bandas rojas: episodios manuales.
      - Bandas azules: episodios automáticos (por estrategia, ventana y métrica).
    """
    vars10 = list(df.columns)[:10]
    rows, cols = 5, 2

    subset_auto = df_episodes_auto[
        (df_episodes_auto["strategy"] == strategy) &
        (df_episodes_auto["window"] == window) &
        (df_episodes_auto["metric"] == metric)
    ]

    fig, axes = plt.subplots(rows, cols, figsize=(12, 16), sharex=True)
    axes = axes.flatten()

    for i, var in enumerate(vars10):
        ax = axes[i]
        s = df[var]

        # Serie de tiempo
        ax.plot(s.index, s.values, linewidth=0.8)
        ax.set_title(var, fontsize=9)

        # Episodios manuales (rojo)
        if show_manual:
            mans = intervals_manual[intervals_manual["variable"] == var]
            for _, m in mans.iterrows():
                ax.axvspan(m["manual_start"], m["manual_end"],
                           alpha=0.18, color="red")

        # Episodios automáticos (azul)
        autos = subset_auto[subset_auto["variable"] == var]
        for _, a in autos.iterrows():
            ax.axvspan(a["seg_start"], a["seg_end"],
                       alpha=0.18, color="blue")

        ax.grid(True, alpha=0.3)

    # Si hay menos de 10 variables, apagar ejes sobrantes
    for j in range(len(vars10), len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(
        f"Episodios — strategy={strategy}, window={window}, metric={metric}\n"
        f"(rojo = manual, azul = auto)",
        fontsize=12
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    return fig

In [18]:
from pathlib import Path

EXPORT_ROOT = OUTPUT_DIR / "plots"
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

PLOT_WINDOWS = ["2H","4H", "6H", "12H", "24H", "36H", "48H"]
PLOT_STRATEGIES   = ["decay","golden","seasonal"]
PLOT_METRICS      = ("psi", "ks", "wasserstein")

print("Exportando combinaciones seleccionadas (metric × strategy × window) a PNG (Matplotlib)")

for win in PLOT_WINDOWS:
    win_dir = EXPORT_ROOT / f"window_{win}"
    win_dir.mkdir(parents=True, exist_ok=True)

    for strat in PLOT_STRATEGIES:
        for metric in PLOT_METRICS:
            subset_auto = df_episodes_auto[
                (df_episodes_auto["strategy"] == strat) &
                (df_episodes_auto["window"]   == win) &
                (df_episodes_auto["metric"]   == metric)
            ]
            if subset_auto.empty:
                continue

            print("\n==============================================")
            print(f"Ventana: {win} | Estrategia: {strat} | Métrica: {metric}")
            print("==============================================")

            fig = plot_grid_episodes_matplotlib(
                df=df,
                df_episodes_auto=df_episodes_auto,
                intervals_manual=intervals_manual,
                strategy=strat,
                window=win,
                metric=metric,
                show_manual=True
            )

            out_path = win_dir / f"episodes_{metric}_{strat}_{win}.png"
            fig.savefig(out_path, dpi=200, bbox_inches="tight")
            plt.close(fig)

            print(f"[OK] Exportado: {out_path}")

Exportando combinaciones seleccionadas (metric × strategy × window) a PNG (Matplotlib)

Ventana: 2H | Estrategia: decay | Métrica: psi
[OK] Exportado: synthetic_data\results\plots\window_2H\episodes_psi_decay_2H.png

Ventana: 2H | Estrategia: decay | Métrica: ks
[OK] Exportado: synthetic_data\results\plots\window_2H\episodes_ks_decay_2H.png

Ventana: 2H | Estrategia: decay | Métrica: wasserstein
[OK] Exportado: synthetic_data\results\plots\window_2H\episodes_wasserstein_decay_2H.png

Ventana: 2H | Estrategia: golden | Métrica: psi
[OK] Exportado: synthetic_data\results\plots\window_2H\episodes_psi_golden_2H.png

Ventana: 2H | Estrategia: golden | Métrica: ks
[OK] Exportado: synthetic_data\results\plots\window_2H\episodes_ks_golden_2H.png

Ventana: 2H | Estrategia: golden | Métrica: wasserstein
[OK] Exportado: synthetic_data\results\plots\window_2H\episodes_wasserstein_golden_2H.png

Ventana: 2H | Estrategia: seasonal | Métrica: psi
[OK] Exportado: synthetic_data\results\plots\window_2H

## 9. Visualizaciones limpias de performance

In [ ]:
if eval_episodes_df.empty:
    print("eval_episodes_df está vacío, no se pueden generar gráficos.")
else:
    # -----------------------------
    # Helpers de orden
    # -----------------------------
    # Orden correcto de ventanas: "2H", "4H", "6H", "12H", "24H", "48H", ...
    window_order = sorted(
        eval_episodes_df["window"].unique(),
        key=lambda w: int(str(w).rstrip("H"))
    )

    # Métricas que realmente queremos mostrar
    metrics_plot = [m for m in ["psi", "ks", "wasserstein"]
                    if m in eval_episodes_df["metric"].unique()]

    # Estrategias en orden
    strategy_order = ["decay", "seasonal", "golden"]

    # =============================
    # 9.1 Heatmaps F1 por métrica
    # =============================
    for metric_name in metrics_plot:
        sub = eval_episodes_df[eval_episodes_df["metric"] == metric_name]

        # heatmap con F1 basado en tiempo
        heat = sub.pivot(index="strategy", columns="window", values="F1_time")

        heat = heat.reindex(index=strategy_order)
        cols_ordered = [w for w in window_order if w in heat.columns]
        heat = heat[cols_ordered]

        fig_heat = px.imshow(
            heat,
            text_auto=".2f",
            aspect="auto",
            color_continuous_scale="Blues",
            title=f"F1_time por estrategia y ventana — métrica = {metric_name}",
            labels=dict(color="F1_time"),


        )
        fig_heat.update_xaxes(title="Ventana")
        fig_heat.update_yaxes(title="Estrategia")
        fig_heat.show()

    # ==========================================
    # 9.2 ¿Cuál es el mejor tamaño de ventana?
    #    (agregado sobre métricas y estrategias)
    # ==========================================
    f1_window = (
        eval_episodes_df
        .groupby("window")["F1"]
        .agg(mean="mean", std="std")
        .reset_index()
        .sort_values("window", key=lambda s: s.str.rstrip("H").astype(int))
    )

    fig_win_bar = px.bar(
        f1_window,
        x="window",
        y="mean",
        error_y="std",
        title="F1 promedio por tamaño de ventana (todas las métricas y estrategias)",
        labels={"window": "Ventana", "mean": "F1 promedio"},
        category_orders={"window": window_order},
    )
    fig_win_bar.show()

    # Distribución completa de F1 por ventana (boxplot)
    fig_win_box = px.box(
        eval_episodes_df,
        x="window",
        y="F1",
        points="all",
        title="Distribución de F1 por ventana (todas las métricas y estrategias)",
        labels={"window": "Ventana", "F1": "F1"},
        category_orders={"window": window_order},
    )
    fig_win_box.show()

    # ==================================================
    # 9.3 Comparación de estrategias y métricas (global)
    # ==================================================
    f1_strat_metric = (
        eval_episodes_df
        .groupby(["metric", "strategy"])["F1"]
        .mean()
        .reset_index()
    )

    fig_strat_metric = px.bar(
        f1_strat_metric,
        x="metric",
        y="F1",
        color="strategy",
        barmode="group",
        category_orders={"metric": metrics_plot, "strategy": strategy_order},
        title="F1 promedio por métrica y estrategia (agregado en todas las ventanas)",
        labels={"metric": "Métrica estadística", "F1": "F1 promedio", "strategy": "Estrategia"},
    )
    fig_strat_metric.show()

    # ========================================================
    # 9.4 Comportamiento por tipo de drift (gradual / abrupto)
    #     usando by_drifttype_manual.csv
    # ========================================================
    # Intentamos usar el DataFrame 'by_type' si existe,
    # si no, lo cargamos desde el CSV exportado.
    by_type_df = None
    if "by_type" in locals():
        by_type_df = by_type.copy()
    else:
        by_type_path = METRICS_DIR / "by_drifttype_manual.csv"
        if by_type_path.exists():
            by_type_df = pd.read_csv(by_type_path)

    if by_type_df is None or by_type_df.empty:
        print("ℹ️ No hay datos por tipo de drift (by_drifttype_manual).")
    else:
        # Agregamos sobre ventanas para comparar MÉTRICA vs ESTRATEGIA
        by_type_agg = (
            by_type_df
            .groupby(["drift_type", "metric", "strategy"])
            .agg(
                Recall=("Recall", "mean"),
                coverage_mean=("coverage_mean", "mean"),
                delay_mean_hours=("delay_mean_hours", "mean"),
            )
            .reset_index()
        )

        # Aseguramos el orden de métricas/estrategias
        by_type_agg["metric"] = pd.Categorical(by_type_agg["metric"],
                                               categories=metrics_plot,
                                               ordered=True)
        by_type_agg["strategy"] = pd.Categorical(by_type_agg["strategy"],
                                                 categories=strategy_order,
                                                 ordered=True)

        # --- 9.4a Recall por tipo de drift ---
        fig_recall = px.bar(
            by_type_agg,
            x="metric",
            y="Recall",
            color="strategy",
            barmode="group",
            facet_col="drift_type",
            category_orders={
                "metric": metrics_plot,
                "strategy": strategy_order,
            },
            title="Recall por métrica y estrategia, separado por tipo de drift",
            labels={"metric": "Métrica estadística", "Recall": "Recall promedio"},
        )
        fig_recall.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
        fig_recall.show()

        # --- 9.4b Cobertura por tipo de drift ---
        fig_cov = px.bar(
            by_type_agg,
            x="metric",
            y="coverage_mean",
            color="strategy",
            barmode="group",
            facet_col="drift_type",
            category_orders={
                "metric": metrics_plot,
                "strategy": strategy_order,
            },
            title="Cobertura promedio por métrica y estrategia, por tipo de drift",
            labels={"metric": "Métrica estadística", "coverage_mean": "Cobertura promedio"},
        )
        fig_cov.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
        fig_cov.show()

        # --- 9.4c Delay por tipo de drift ---
        fig_delay = px.bar(
            by_type_agg,
            x="metric",
            y="delay_mean_hours",
            color="strategy",
            barmode="group",
            facet_col="drift_type",
            category_orders={
                "metric": metrics_plot,
                "strategy": strategy_order,
            },
            title="Delay promedio (h) por métrica y estrategia, por tipo de drift",
            labels={"metric": "Métrica estadística",
                    "delay_mean_hours": "Delay medio (horas)"},
        )
        fig_delay.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
        fig_delay.show()
